# Notebook #2c: ALA & ACRL Cross-Comparisons

## 1.0 Project Dependencies

### 1.1 Pip Installs

### 1.2 Import Libraries

In [1]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly as py
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

### 1.3 Import Datasets

In [2]:
# ACRL
acrl_google       = pd.read_parquet("Cleaned Datasets/acrl_google")
acrl_indiv_emails = pd.read_parquet("Cleaned Datasets/acrl_indiv_emails")
acrl_syllabus     = pd.read_parquet("Cleaned Datasets/acrl_syllabus")

# ALA Editions
editions_eblasts     = pd.read_parquet("Cleaned Datasets/editions_ala_eblasts.parquet")
editions_fw26_google = pd.read_parquet("Cleaned Datasets/editions_ala_fw26_google.parquet")
editions_ss25_google = pd.read_parquet("Cleaned Datasets/editions_ala_ss25_google.parquet")
editions_ala_stats   = pd.read_parquet("Cleaned Datasets/editions_ala_stats_eblasts")

### 1.4 Standardize Setup & Styling

In [3]:
ALA_BLUE   = '#003F72'
ALA_RED    = '#C8102E'
ACRL_TEAL  = '#00857C'
ACCENT     = '#F5A623'
LIGHT_GRAY = '#F4F4F4'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   LIGHT_GRAY,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'sans-serif',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

### 1.5 Global Variables

In [4]:
# ── Light type coercion ─────────────────────────────────────────────────────
# editions_ala_stats: parse Date, strip whitespace from pct columns if needed
editions_ala_stats['Date'] = pd.to_datetime(editions_ala_stats['Date'], errors='coerce')

# acrl_google: parse dates
acrl_google['date_start'] = pd.to_datetime(acrl_google['date_start'], errors='coerce')
acrl_google['date_end']   = pd.to_datetime(acrl_google['date_end'],   errors='coerce')

# Editions Google: ensure start_date / end_date are datetime
for df in [editions_ss25_google, editions_fw26_google]:
    df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
    df['end_date']   = pd.to_datetime(df['end_date'],   errors='coerce')

# Isolate homepage rows for time-series work
ss25_home = editions_ss25_google[editions_ss25_google['page_path_and_screen_class'] == '/SS25catalog'].copy()
fw26_home = editions_fw26_google[editions_fw26_google['page_path_and_screen_class'] == '/FW26catalog'].copy()

print('Type coercion complete.')

Type coercion complete.
/tmp/ipykernel_1371/921417517.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  editions_ala_stats['Date'] = pd.to_datetime(editions_ala_stats['Date'], errors='coerce')


<hr>

## 2.0 ALA & ACRL Summary Statistics

### 2.1 Side-by-Side Comparison

In [5]:
# Compare ACRL vs ALA Editions aggregate traffic side-by-side
acrl_row = acrl_google.iloc[0]

comparison = pd.DataFrame({
    'Metric': ['Total Views', 'Total Active Users', 'Views/User', 'Avg Engagement (s)'],
    'ACRL Catalog (Jan 25–Feb 26)': [
        acrl_row['total_views'],
        acrl_row['active_users'],
        round(acrl_row['views_per_active_user'], 2),
        acrl_row['avg_engagement_time_seconds_per_active_user'],
    ],
    'ALA Editions SS25': [
        ss25_home['views'].sum(),
        ss25_home['active_users'].sum(),
        round(ss25_home['views_per_active_user'].mean(), 2),
        round(ss25_home['average_engagement_time_per_active_user'].mean(), 1),
    ],
    'ALA Editions FW26': [
        fw26_home['views'].sum(),
        fw26_home['active_users'].sum(),
        round(fw26_home['views_per_active_user'].mean(), 2),
        round(fw26_home['average_engagement_time_per_active_user'].mean(), 1),
    ],
}).set_index('Metric')

display(comparison)

,ACRL Catalog (Jan 25–Feb 26),ALA Editions SS25,ALA Editions FW26
Metric,,,
Total Views,4784.00,10208.0,46622.00
Total Active Users,3345.00,8203.0,26402.00
Views/User,1.43,1.3,1.53
Avg Engagement (s),47.00,23.7,19.60


### 2.2 Bar Chart Comparison

In [6]:
metrics_to_plot = ['Total Views', 'Total Active Users']
comp_sub = comparison.loc[metrics_to_plot]

# 1. Data Setup (using the comp_sub and metrics_to_plot from your cell)
catalogs = comp_sub.columns.tolist()

# Updated colors: Teal, Blue, and the brand Gold/Accent instead of Red
colors = [ACRL_TEAL, ALA_BLUE, ACCENT] 

fig = go.Figure()

for cat, color in zip(catalogs, colors):
    fig.add_trace(go.Bar(
        x=metrics_to_plot,
        y=comp_sub[cat],
        name=cat,
        marker_color=color,
        # Adding labels on top of bars
        text=comp_sub[cat],
        textposition='outside',
        texttemplate='%{text:,.0f}', # Formats with commas (e.g., 46,622)
        cliponaxis=False
    ))

# 2. Update Layout & Styling
fig.update_layout(
    title_text="<b>ACRL vs. ALA Editions — Website Traffic Volume Comparison</b>",
    title_x=0.5,
    xaxis_title="Metric",
    yaxis_title="Count",
    barmode='group',
    template="plotly_white",
    height=650,
    width=900,
    # Clean legend at the top
    legend=dict(
        orientation="h",
        yanchor="top",
        y=1.1,
        xanchor="left",
        x=0.13
    ),
    margin=dict(t=120) # Extra room for top legend/labels
)

# Soften gridlines
fig.update_yaxes(gridcolor='whitesmoke')

fig.show()

<hr>

## 3.0 ALA & ACRL Email Subject Line Analysis

In [7]:
# Strip text of repetative and noisy strings of text
PREFIXES_TO_STRIP = [
    r"^New from ACRL\s*[-–:]\s*",
    r"^The Syllabus From ACRL\s*[-–:]\s*",
    r"^ACRL releases\s+",
    r"^New edition\s*[-–:]+\s*",
    r"^OUT NOW:\s*",
]

def strip(text):
    for pat in PREFIXES_TO_STRIP:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*new\s+", "", text, flags=re.IGNORECASE)  # catch leftover "new"
    return text.strip().strip('"\u201c\u201d')

In [8]:
# Prepare a dataframe for the subject line analysis
acrl_subject_line = pd.concat([acrl_indiv_emails, acrl_syllabus], axis=0)
acrl_subject_line = acrl_subject_line[['Subject', 'Open %', 'Folder']]

ala_subject_line = editions_ala_stats[['Subject', 'Opened %']].copy()
ala_subject_line['Folder'] = 'ALA'
ala_subject_line = ala_subject_line.rename(columns={'Opened %': 'Open %'})
ala_subject_line = ala_subject_line[['Subject', 'Open %', 'Folder']]

subject_line = pd.concat([ala_subject_line, acrl_subject_line], axis=0)
subject_line['Subject'] = subject_line['Subject'].apply(strip)
subject_line

,Subject,Open %,Folder
0,Celebrating school libraries,17.40,ALA
1,Free October 16th webinar - Programming and Co...,17.00,ALA
2,Advocacy resources for public libraries,19.30,ALA
3,Nuture a joy-positive library,15.40,ALA
4,Free Zoom event: 72 Tips for Academic Librarie...,18.60,ALA
...,...,...,...
16,October 2024,20.62,ACRL
17,September 2024,20.30,ACRL
18,August 2024,21.60,ACRL
19,July 2024,24.80,ACRL


In [9]:
def run_tfidf(df, min_appearances=2, ngram_range=(1,2)):
    """
    For each term found by TF-IDF, compute the average open rate
    of all campaigns where that term appears.
    """
    MONTHS = {"january","february","march","april","may","june",
          "july","august","september","october","november","december"}

    custom_stop_words = list(ENGLISH_STOP_WORDS | MONTHS)
    
    vec = TfidfVectorizer(
        ngram_range=ngram_range,
        stop_words=custom_stop_words,
        min_df=min_appearances
    )
    X = vec.fit_transform(df["Subject"])
    terms = vec.get_feature_names_out()

    rows = []
    for i, term in enumerate(terms):
        present = X[:, i].toarray().flatten() > 0 
        if present.sum() < min_appearances:
            continue
        rows.append({
            "term":         term,
            "avg_open_pct": round(df.loc[present, "Open %"].mean(), 1),
            "appearances":  int(present.sum()),

        })
    return pd.DataFrame(rows).sort_values("avg_open_pct", ascending=False)

In [10]:
# Run the TFIDF vectorizer on the Subject Line Data
baseline = round(subject_line["Open %"].mean(), 1)

term_df = run_tfidf(subject_line)
print(term_df.head(15).to_string(index=False))
print(term_df.tail(15).to_string(index=False))

             term  avg_open_pct  appearances
            store          34.8            2
        ala store          34.8            2
     perspectives          34.0            2
       supporting          33.8            2
          success          33.6            2
         students          33.4            2
community college          33.2            2
          workers          32.8            3
  library workers          32.8            3
  college library          31.6            3
          college          31.6            3
       strategies          31.6            2
 academic library          30.6            3
            books          29.5            3
        books ala          29.5            3
            term  avg_open_pct  appearances
              ai          20.4            6
         catalog          20.4            6
            free          19.9            7
        literacy          19.7            6
  school library          18.6            2
         digital

In [11]:
# 1. Prepare Data
top15 = term_df.head(15).sort_values('avg_open_pct', ascending=True)
bot15 = term_df.tail(15).sort_values('avg_open_pct', ascending=True)

# 2. Create Subplots
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("<b>Top 15 Terms</b>", "<b>Bottom 15 Terms</b>"),
    horizontal_spacing=0.15
)

# 3. Add Top 15 (ALA Blue)
fig.add_trace(
    go.Bar(
        y=top15['term'],
        x=top15['avg_open_pct'],
        orientation='h',
        marker_color=ALA_BLUE,
        name='Top Terms',
        text=top15.apply(lambda r: f"{r['avg_open_pct']}% (n={int(r['appearances'])})", axis=1),
        textposition='outside'
    ),
    row=1, col=1
)

# 4. Add Bottom 15 (Brand Accent/Gold)
fig.add_trace(
    go.Bar(
        y=bot15['term'],
        x=bot15['avg_open_pct'],
        orientation='h',
        marker_color=ACCENT, 
        name='Bottom Terms',
        text=bot15.apply(lambda r: f"{r['avg_open_pct']}% (n={int(r['appearances'])})", axis=1),
        textposition='outside'
    ),
    row=1, col=2
)

# 5. Add Baseline Legend Entry (Dummy Trace)
fig.add_trace(
    go.Scatter(
        x=[None], y=[None],
        mode='lines',
        line=dict(color='black', width=1.5, dash='dash'),
        name=f'Baseline ({baseline}%)'
    )
)

# 6. Add the actual vertical lines (without overlapping text)
for i in [1, 2]:
    fig.add_vline(x=baseline, line_dash="dash", line_color="black", line_width=1.5, row=1, col=i)

# 7. Update Layout with Centered Legend
fig.update_layout(
    title_text="<b>Subject Line Terms vs. Avg Open Rate (TF-IDF)</b>",
    title_x=0.5,
    showlegend=True,
    template="plotly_white",
    height=600, width=1100,
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.06,
        xanchor="center", x=0.45
    ),
    margin=dict(t=120, b=50, l=150, r=50)
)

fig.update_xaxes(title_text="Avg Open Rate (%)", range=[0, term_df['avg_open_pct'].max() + 12])

fig.show()

## 4.0 ALA & ACRL Body Text Analysis

In [12]:
# Prepare the text data (email body text)
bodies = editions_ala_stats['Body Text'].fillna('')

# Initialize TF-IDF Vectorizer with English stop words, limiting to top 20 features
vectorizer = TfidfVectorizer(stop_words='english', max_features=20)

# Fit and transform the bodies
tfidf_matrix = vectorizer.fit_transform(bodies)

# Get feature names and mean TF-IDF scores across all documents
feature_names = vectorizer.get_feature_names_out()
tfidf_scores = tfidf_matrix.mean(axis=0).A1

# Create a DataFrame for top words
top_words_ala = pd.DataFrame({
    'word': feature_names,
    'score': tfidf_scores
}).sort_values('score', ascending=False)

# Comparison of TF-IDF terms between ALA and ACRL body text
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ACRL body text TF-IDF (using high performers as representative)
acrl_bodies = acrl_indiv_emails['body_text'].fillna('')
vectorizer_acrl = TfidfVectorizer(stop_words='english', max_features=15)
tfidf_acrl = vectorizer_acrl.fit_transform(acrl_bodies)
scores_acrl = tfidf_acrl.mean(axis=0).A1
top_acrl = pd.DataFrame({
    'word': vectorizer_acrl.get_feature_names_out(),
    'score': scores_acrl
}).sort_values('score', ascending=False).head(10)

# ALA top 10
top_ala = top_words_ala.head(10)

# Create comparison subplots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('ALA Body Text TF-IDF', 'ACRL Body Text TF-IDF'),
    horizontal_spacing=0.1
)

fig.add_trace(
    go.Bar(
        x=top_ala['score'],
        y=top_ala['word'],
        orientation='h',
        marker_color=ALA_BLUE,
        name='ALA'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=top_acrl['score'],
        y=top_acrl['word'],
        orientation='h',
        marker_color=ACRL_TEAL,
        name='ACRL'
    ),
    row=1, col=2
)

fig.update_layout(
    title='<b>Comparison of Top TF-IDF Terms: ALA vs. ACRL Email Body Text</b>', title_x=0.5,
    height=600,
    width=1200,
    showlegend=False
)

fig.update_yaxes(autorange='reversed', row=1, col=1)
fig.update_yaxes(autorange='reversed', row=1, col=2)

fig.show()